### 0. Initial Setup: Drive & Dependencies
Before starting the pipeline, we mount Google Drive to access the project files and install the necessary libraries.

In [ ]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Define and change to the project root directory
PROJECT_ROOT_PATH = '/content/drive/MyDrive/Colab Notebooks/Senior Project'
os.chdir(PROJECT_ROOT_PATH)

print(f"Current working directory: {os.getcwd()}")

In [ ]:
# Install dependencies from requirements.txt
if os.path.exists('requirements.txt'):
    !pip install -r requirements.txt
else:
    print("requirements.txt not found in the current directory.")

In [ ]:
!pip install "https://github.com/lesj0610/flash-attention/releases/download/v2.8.3-cu12-torch2.11/flash_attn-2.8.3+cu12torch2.11cxx11abiTRUE-cp312-cp312-linux_x86_64.whl"

# Omni-Phi: End-to-End Speech-to-Speech Translation Pipeline

This notebook demonstrates the end-to-end training and inference workflow for the **Omni-Phi** S2ST model. Omni-Phi translates spoken English to spoken German directly, bypasses intermediate text decoding, and leverages the frozen EnCodec codes combined with the **microsoft/Phi-4-multimodal-instruct** model's audio capabilities via a speech LoRA adapter.

### Pipeline Stages:
1. **Environment Setup & Path Configuration**
2. **Phase 1: Run Target Tokenization & Preprocessing**
3. **Phase 2 & 3: Load Multimodal Model, Processor, & Dataset**
4. **Phase 4: Run Training Loop using Custom Data Collator**
5. **Phase 5: Run Speech-to-Speech Translation Inference**

## 1. Environment Setup & Path Configuration

First, we import core dependencies and add the project root to `sys.path` so we can import our modules (`dataset`, `model`, `collator`, `inference`, `encoders`, etc.).

In [ ]:
import os
import sys
from pathlib import Path

# Resolve directories relative to the current notebook location
notebook_dir = Path(os.getcwd()).resolve()
project_root = notebook_dir
omni_phi_dir = project_root / "models" / "omni_phi"

# Add paths to sys.path
# We add both the root and the omni_phi subdirectory to ensure all local modules are findable
paths_to_add = [str(project_root), str(omni_phi_dir)]
for path in paths_to_add:
    if path not in sys.path:
        sys.path.append(path)

print(f"Project root  : {project_root}")
print(f"Omni Phi path : {omni_phi_dir}")
print(f"Active paths in sys.path: {[p for p in sys.path if 'Senior Project' in p]}")

## 2. Phase 1: Preprocessing & Tokenization

Before training, we must extract source waveforms and encode target audios into interleaved EnCodec codebook tokens offset by `+100,000` to fit the Phi-4 vocabulary.

We can trigger Phase 1 preprocessing directly from the notebook.

> **Note:** For demonstration purposes, we will preprocess a small subset of the `fleurs` dataset. Adjust `--num_samples` as needed for full training.

In [ ]:
# Run Phase 1 Preprocessing script
!python "{project_root}/models/omni_phi/preprocess_omni.py" \
    --dataset fleurs \
    --split train \
    --num_samples 15000 \
    --bandwidth 1.5 \
    --token_offset 100000

## 3. Phase 2 & 3: Load Model, Processor, & Dataset

Now we import our custom dataset class (`OmniPhiDataset`) and core model wrapper (`OmniPhiS2ST`). We will load the underlying multimodal processor, prepare datasets from preprocessed JSONL files, and load our wrapper with the LoRA speech adapter active.

In [ ]:
import torch
from transformers import AutoProcessor
from dataset import OmniPhiDataset
from model import OmniPhiS2ST

checkpoint_output_dir = omni_phi_dir / "checkpoints_en_de"
PHI4_HUB_ID = "microsoft/Phi-4-multimodal-instruct"

# MODEL_ID = "microsoft/Phi-4-multimodal-instruct"    # Start from the fresh weights
MODEL_ID = str(checkpoint_output_dir)                 # Resume from last checkpoint
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

# 1. Load Processor
print("Loading AutoProcessor...")
processor = AutoProcessor.from_pretrained(PHI4_HUB_ID, trust_remote_code=True)

# 2. Load custom training dataset and evaluation dataset
data_path = omni_phi_dir / "data" / "preprocessed"
train_jsonl = data_path / "train.jsonl"

print(f"Loading dataset from {train_jsonl}...")
# train_dataset = OmniPhiDataset(str(train_jsonl), processor, training=True)

# 3. Load Omni-Phi Model Wrapper (with frozen EnCodec and trainable LoRA speech adapter)
print("Loading model wrapper...")
model = OmniPhiS2ST(phi4_model_id=MODEL_ID, device=device)
model.processor = processor


## 4. Phase 4: Data Collator & Training Loop

With our custom `omni_phi_collate_fn` import, we pad multimodal batch sequences leftwards while masking the prompt region (`labels = -100`), ensuring the model learns exclusively to predict target interleaved EnCodec token sequences.

In [ ]:
import torch
from collator import omni_phi_collate_fn
from trainer import OmniPhiTrainer
from transformers import TrainingArguments

# =========================================================================
# PHASE 1: PREPARE MODEL & SELECT ADAPTER
# =========================================================================
model.phi4.set_lora_adapter('speech')

# Enforce freezing and trainable parameters strictly BEFORE trainer initialization
lora_count = 0
for name, param in model.phi4.named_parameters():
    if 'lora' in name.lower() or 'embed_tokens' in name.lower() or 'lm_head' in name.lower():
        param.requires_grad_(True)
        if 'lora' in name.lower():
            lora_count += 1
    else:
        param.requires_grad_(False)

# Force base model input require grads hook (which works in tandem with our model.py wrapper fix!)
model.phi4.enable_input_require_grads()
model.phi4.config.use_cache = False

# Quick verification checks
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"✅ {lora_count} LoRA tensors marked trainable ({trainable_params:,} params)")
assert trainable_params > 0, "STOP: still no trainable params — do not proceed"

# =========================================================================
# PHASE 2: CONFIGURE ROBUST TRAINING ARGUMENTS
# =========================================================================
# Enable TF32 matrix calculations for speed boosts on NVIDIA A100 GPU
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

_optim = "adamw_torch_fused" if torch.cuda.is_available() else "adamw_torch"

training_args = TrainingArguments(
    output_dir                    = str(checkpoint_output_dir),
    num_train_epochs              = 1,
    per_device_train_batch_size   = 4,            # keep — VRAM constrained
    gradient_accumulation_steps   = 4,            # effective batch 16
    gradient_checkpointing        = True,         # Kept active for optimization & VRAM safety
    gradient_checkpointing_kwargs = {"use_reentrant": False}, # Set to non-reentrant for stable custom pipelines

    # Optimizer
    optim         = _optim,
    adam_beta1    = 0.9,
    adam_beta2    = 0.95,
    adam_epsilon  = 1e-7,
    learning_rate = 4e-5,
    weight_decay  = 0.01,
    max_grad_norm = 5.0,

    # Scheduler
    lr_scheduler_type = "constant_with_warmup",
    warmup_ratio      = 0.01,

    # Precision
    bf16  = True,
    tf32  = True,
    fp16  = False,

    # Logging & saving
    logging_steps  = 10,
    save_strategy  = "steps",
    save_steps=100,
    save_safetensors = False,

    # DataLoader Settings
    dataloader_num_workers  = 4,
    dataloader_pin_memory   = True,

    remove_unused_columns    = False,
    report_to                = "none",
    ddp_find_unused_parameters = True,
)

# =========================================================================
# PHASE 3: INIT TRAINER & START TRAINING
# a
# The trainer now inspects the fully pre-configured model correctly
trainer = OmniPhiTrainer(
    model=model,
    args=training_args,
    data_collator=omni_phi_collate_fn,
    train_dataset=train_dataset,
)

print("Starting training...")
trainer.train(
    # resume_from_checkpoint=True
    )

# Save model weights
trainer.save_model(str(checkpoint_output_dir))
print(f"✅ Training complete. Checkpoint saved to {checkpoint_output_dir}")


In [ ]:
try:
    model.processor.save_pretrained(str(checkpoint_output_dir))
except AttributeError:
    model.processor.tokenizer.save_pretrained(str(checkpoint_output_dir))
    print("✅ Tokenizer saved successfully.")


## 5. Phase 5: Inference — Strategy Comparison

Three ways to get audio out of the model:

| # | Method | Best for |
|---|--------|----------|
| A | `model.generate_speech(audio, max_new_tokens=800)` | Evaluation / sentence-length inputs |
| B | `translate_speech(wav_path, model=model, ...)` | File-level convenience wrapper (same as A internally) |
| C | `translate_speech_batched(model, audio, ...)` | Long-form audio (OOM safety — **not** recommended for FLEURS sentences) |

> **How does `generate_speech` know the task?**  
> The prompt is hardcoded in `model.py` as:  
> ```
> <|audio_1|>\nTranslate this to spoken german:
> ```  
> This is the **exact string used during fine-tuning** (`INSTRUCTION` in `dataset.py`).  
> The LoRA weights learned to respond to this specific prompt with EnCodec audio tokens.  
> Changing the prompt at inference time will produce garbled output.

In [ ]:
# ── Inference Comparison: imports ────────────────────────────────────────
import numpy as np
import soundfile as sf
import tempfile, os, librosa
from IPython.display import Audio, display
from transformers import pipeline as hf_pipeline
from inference import translate_speech, translate_speech_batched

print("Inference comparison imports ready.")
model.eval()
print(f"Model: {type(model).__name__} | device: {model.device}")

In [ ]:
import numpy as np
from dataset_loader import load_data
from IPython.display import Audio, display

# Load 1 sample so this cell runs in seconds
data = load_data(lang=["en"], split="train", num_samples=100, dataset=["fleurs"])
sample = data["en"][0]

en_audio = np.array(sample["audio"]["array"], dtype=np.float32)
en_sr    = sample["audio"]["sampling_rate"]
en_text  = sample.get("transcription", "<no transcription>")

print(f"Sample SR    : {en_sr} Hz")
print(f"Duration     : {len(en_audio)/en_sr:.2f} s")
print(f"Transcription: {en_text}")
display(Audio(en_audio, rate=en_sr))

In [ ]:
# ── METHOD A: model.generate_speech ─────────────────────────────────────
# 800 tokens @ 150 tok/s (75 frames/s × 2 codebooks at 1.5 kbps) ≈ 5.3 s.
# generate_speech uses the EXACT training prompt internally:
#   '<|audio_1|>\nTranslate this to spoken german:'
# This is hardcoded in model.py to match dataset.py's INSTRUCTION constant.

WAV_A = "translated_method_A.wav"

print("[A] Running model.generate_speech(max_new_tokens=800)...")
audio_A = model.generate_speech(en_audio, source_sr=en_sr, max_new_tokens=800)
sf.write(WAV_A, audio_A, samplerate=24000)
print(f"[A] Generated {len(audio_A)/24000:.2f} s → saved to {WAV_A}")
display(Audio(audio_A, rate=24000))

In [ ]:
# ── METHOD B: translate_speech (file-path convenience wrapper) ───────────
# Internally calls model.generate_speech with the same prompt.
# We override max_new_tokens=800 to get a fair comparison.

WAV_B   = "translated_method_B.wav"
WAV_B_IN = "source_english.wav"

# Save source to a temp input file (translate_speech needs a path)
sf.write(WAV_B_IN, en_audio, en_sr)

print("[B] Running translate_speech(max_new_tokens=800)...")
audio_B = translate_speech(
    source_wav_path=WAV_B_IN,
    output_wav_path=WAV_B,
    model=model,
    max_new_tokens=800,
)
print(f"[B] Generated {len(audio_B)/24000:.2f} s → saved to {WAV_B}")
display(Audio(audio_B, rate=24000))

In [ ]:
# ── METHOD C: translate_speech_batched ───────────────────────────────────
# Runs generate_speech in chunks of BATCH_TOKEN_LIMIT (200 tok ≈ 1.3 s) and
# CONCATENATES them. For sentence-length audio each chunk is an independent
# re-translation, so the output will likely be repetitive/garbled.
# This method is designed for very long recordings that would OOM in one pass.

WAV_C = "translated_method_C.wav"

print("[C] Running translate_speech_batched (max_batches=5)...")
audio_C = translate_speech_batched(model, en_audio, source_sr=en_sr, max_batches=5)
sf.write(WAV_C, audio_C, samplerate=24000)
print(f"[C] Generated {len(audio_C)/24000:.2f} s → saved to {WAV_C}")
display(Audio(audio_C, rate=24000))

In [ ]:
import librosa

asr = hf_pipeline("automatic-speech-recognition", model="openai/whisper-base", device=device)

def transcribe(audio_24k, label):
    audio_16k = librosa.resample(audio_24k, orig_sr=24000, target_sr=16000)
    result = asr({"array": audio_16k, "sampling_rate": 16000},
                 generate_kwargs={"language": "de"})
    print(f"[{label}] {result['text'].strip()}")
    return result['text'].strip()

print("=" * 70)
print(f"  Source (EN): {en_text}")
print("=" * 70)
t_A = transcribe(audio_A, "A  generate_speech(max=800)   ")
t_B = transcribe(audio_B, "B  translate_speech(max=800)  ")
t_C = transcribe(audio_C, "C  translate_speech_batched() ")
print("=" * 70)

In [ ]:
import pandas as pd

rows = [
    {"Method": "A: generate_speech(max_new_tokens=800)",
     "Duration (s)": f"{len(audio_A)/24000:.2f}",
     "ASR transcription": t_A,
     "Notes": "Recommended for eval — one clean pass"},
    {"Method": "B: translate_speech(max_new_tokens=800)",
     "Duration (s)": f"{len(audio_B)/24000:.2f}",
     "ASR transcription": t_B,
     "Notes": "File wrapper; same as A under the hood"},
    {"Method": "C: translate_speech_batched(max_batches=5)",
     "Duration (s)": f"{len(audio_C)/24000:.2f}",
     "ASR transcription": t_C,
     "Notes": "Repeats source each chunk — expect garbled output for sentences"},
]
pd.set_option("display.max_colwidth", 80)
pd.DataFrame(rows)

In [ ]:
# ── Sanity check: Whisper transcription of all three outputs ─────────────
# Uses CPU + int8 so Phi-4 can stay on GPU without OOM.

from faster_whisper import WhisperModel
import soundfile as sf
import numpy as np

whisper = WhisperModel("tiny", device="cpu", compute_type="int8")

OUTPUTS = [
    ("A", "model.generate_speech",     WAV_A),
    ("B", "translate_speech",           WAV_B),
    ("C", "translate_speech_batched",   WAV_C),
]

print(f"Source (EN): {en_text}")
print("=" * 70)

for label, method_name, wav_path in OUTPUTS:
    if not os.path.exists(wav_path):
        print(f"[{label}] {wav_path} not found — skipping")
        continue

    audio, sr = sf.read(wav_path)
    audio = audio.astype(np.float32)

    # German transcription
    segments, info = whisper.transcribe(audio, language="de", beam_size=5, vad_filter=True)
    de_text = " ".join(seg.text.strip() for seg in segments)

    # Auto-detect language (confidence check)
    _, info2 = whisper.transcribe(audio, beam_size=5, vad_filter=True)

    print(f"[{label}] {method_name}")
    print(f"    DE transcription : {de_text}")
    print(f"    Lang confidence  : {info.language_probability:.1%} (forced DE)")
    print(f"    Auto-detected    : {info2.language} ({info2.language_probability:.1%})")
    print()


In [ ]:
from google.colab import runtime
# Terminate the runtime
runtime.unassign()